## NOAA flood data 1996-most recent for 10 states

In [76]:
import pandas as pd
import requests
import re
import os
import geopandas as gpd
import zipfile
import io
import fiona
from shapely.geometry import Polygon, MultiPolygon
import shapely
import pyogrio
import shutil

In [77]:
state_epsg = {
    "01": 26916,  
    "05": 26915, 
    "19": 26915, 
    "21": 26916, 
    "28": 26915,  
    "29": 26915, 
    "31": 26914, 
    "40": 26914,  
    "47": 26916, 
    "54": 26917   
}

layer = layer.to_crs(
    epsg=state_epsg[state_fips]
)

In [78]:
project_dir = "/Users/sophiameyers/Documents/GitHub/USGS-Gages-Research"

output_folder = os.path.join(
    project_dir,
    "data",
    "intermediate"
)

os.makedirs(output_folder, exist_ok=True)

In [79]:
base_url = (
    "https://www.ncei.noaa.gov/pub/data/swdi/"
    "stormevents/csvfiles/"
)

states = [
    "WEST VIRGINIA",
    "ALABAMA",
    "ARKANSAS",
    "IOWA",
    "KENTUCKY",
    "MISSISSIPPI",
    "MISSOURI",
    "NEBRASKA",
    "OKLAHOMA",
    "TENNESSEE"
]

response = requests.get(base_url, timeout=60)
response.raise_for_status()

files = re.findall(
    r'href="(StormEvents_details-ftp_v1\.0_d\d{4}_c\d+\.csv\.gz)"',
    response.text
)

year_files = {}

for filename in files:
    year = int(re.search(r"_d(\d{4})_", filename).group(1))
    if year >= 1996:
        year_files[year] = filename

print("Available years:", min(year_files), "-", max(year_files))
print("Files to download:", len(year_files))

flood_data = []

for year, filename in sorted(year_files.items()):

    url = base_url + filename
    print(f"Downloading and filtering {year}...")

    yearly_data = pd.read_csv(
        url,
        compression="gzip",
        low_memory=False
    )

    filtered = yearly_data[
        yearly_data["STATE"].str.strip().str.upper().isin(states) &
        yearly_data["EVENT_TYPE"].str.contains(
            "FLOOD", case=False, na=False
        )
    ].copy()

    flood_data.append(filtered)

noaa_floods = pd.concat(
    flood_data,
    ignore_index=True
)

noaa_floods = noaa_floods.sort_values(
    ["BEGIN_DATE_TIME", "STATE"]
).reset_index(drop=True)

os.makedirs("data/intermediate", exist_ok=True)

output_file = (
    "data/intermediate/NOAA_Flood_Events_10_States_1996_2026.csv"
)

noaa_floods.to_csv(output_file, index=False)

# 
print("\nDOWNLOAD COMPLETE")
print("Total flood records:", len(noaa_floods))
print("States:", noaa_floods["STATE"].nunique())
print("Years:", noaa_floods["YEAR"].min(), "-", noaa_floods["YEAR"].max())

print("\nRecords by state:")
display(
    noaa_floods.groupby("STATE").size()
    .sort_values(ascending=False)
    .to_frame("flood_records")
)

print("\nRecords by event type:")
display(
    noaa_floods.groupby("EVENT_TYPE").size()
    .sort_values(ascending=False)
    .to_frame("records")
)

print("\nSaved to:", output_file)


Available years: 1996 - 2026
Files to download: 31

DOWNLOAD COMPLETE
Total flood records: 54898
States: 10
Years: 1996 - 2026

Records by state:


,flood_records
STATE,
MISSOURI,10830
KENTUCKY,8324
IOWA,7389
ARKANSAS,5693
TENNESSEE,4493
OKLAHOMA,4347
WEST VIRGINIA,4243
MISSISSIPPI,3820
ALABAMA,3389



Records by event type:


,records
EVENT_TYPE,
Flash Flood,34131
Flood,20710
Coastal Flood,57



Saved to: data/intermediate/NOAA_Flood_Events_10_States_1996_2026.csv


### now have NOAA data saved to data/intermediate for the 10 states called NOAA_Flood_Events_10_States_1996_2026.csv

#### Figure out the best way to aggregate it spatially? Pull in census api to amend county boundaries to this output and export it as a geodataframe. 

## Amending a county geometry to each row

In [80]:
events = noaa_floods.copy()

events["STATE_FIPS"] = (
    pd.to_numeric(events["STATE_FIPS"], errors="coerce")
    .astype("Int64")
    .astype("string")
    .str.zfill(2)
)

events["CZ_FIPS"] = (
    pd.to_numeric(events["CZ_FIPS"], errors="coerce")
    .astype("Int64")
    .astype("string")
    .str.zfill(3)
)

events["COUNTY_FIPS"] = (
    events["STATE_FIPS"] + events["CZ_FIPS"]
)

county_events = events[
    events["CZ_TYPE"] == "C"
].copy()

zone_events = events[
    events["CZ_TYPE"] == "Z"
].copy()

print("County events:", len(county_events))
print("Forecast-zone events:", len(zone_events))

County events: 49552
Forecast-zone events: 5346


### Downloading county shapes to data/raw as tiger_counties

In [81]:

url = "https://www2.census.gov/geo/tiger/TIGER2025/COUNTY/tl_2025_us_county.zip"

print("Downloading county boundaries...")

response = requests.get(url, timeout=180)
response.raise_for_status()

output_folder = "data/raw/tiger_counties"
os.makedirs(output_folder, exist_ok=True)

with zipfile.ZipFile(io.BytesIO(response.content)) as z:
    z.extractall(output_folder)

counties = gpd.read_file(
    os.path.join(output_folder, "tl_2025_us_county.shp")
)

print("County polygons downloaded:", len(counties))
display(counties.head())

County polygons downloaded: 3235


,STATEFP,COUNTYFP,COUNTYNS,GEOID,GEOIDFQ,NAME,NAMELSAD,LSAD,CLASSFP,MTFCC,CSAFP,CBSAFP,METDIVFP,FUNCSTAT,ALAND,AWATER,INTPTLAT,INTPTLON,geometry
0,40,075,01101825,40075,0500000US40075,Kiowa,Kiowa County,06,H1,G4020,NaN,NaN,NaN,A,2629039892,40296743,+34.9214893,-098.9816168,"POLYGON ((-98.95506 35.11643, -98.94903 35.116..."
1,46,079,01265776,46079,0500000US46079,Lake,Lake County,06,H1,G4020,NaN,NaN,NaN,A,1457916151,31746795,+44.0284497,-097.1232229,"POLYGON ((-96.88886 43.9353, -96.88886 43.9351..."
2,37,033,01008542,37033,0500000US37033,Caswell,Caswell County,06,H1,G4020,NaN,NaN,NaN,A,1102042927,8293623,+36.3943252,-079.3396193,"POLYGON ((-79.14343 36.4422, -79.14345 36.4418..."
3,48,377,01383974,48377,0500000US48377,Presidio,Presidio County,06,H1,G4020,NaN,NaN,NaN,A,9985057447,1773188,+30.0058912,-104.2616192,"POLYGON ((-104.98078 30.62552, -104.98073 30.6..."
4,39,057,01074041,39057,0500000US39057,Greene,Greene County,06,H1,G4020,212,19430,NaN,A,1071302625,6798109,+39.6874785,-083.8948943,"POLYGON ((-84.10668 39.68891, -84.10662 39.689..."


In [82]:
state_fips = [
    "01", "05", "19", "21", "28",
    "29", "31", "40", "47", "54"
]

counties = counties[
    counties["STATEFP"].isin(state_fips)
].copy()

counties["GEOID"] = (
    counties["STATEFP"] + counties["COUNTYFP"]
)

counties = counties[
    ["GEOID", "NAME", "STATEFP", "COUNTYFP", "geometry"]
]

print("County polygons:", len(counties))
display(counties.head())

County polygons: 878


,GEOID,NAME,STATEFP,COUNTYFP,geometry
0,40075,Kiowa,40,075,"POLYGON ((-98.95506 35.11643, -98.94903 35.116..."
8,40023,Choctaw,40,023,"POLYGON ((-95.99158 34.1568, -95.99154 34.1568..."
12,40129,Roger Mills,40,129,"POLYGON ((-99.61073 35.84756, -99.61003 35.848..."
15,01075,Lamar,01,075,"POLYGON ((-88.24147 33.80335, -88.24125 33.804..."
17,47117,Marshall,47,117,"POLYGON ((-86.73067 35.26956, -86.73072 35.269..."


In [83]:
noaa_spatial = county_events.merge(
    counties,
    left_on="COUNTY_FIPS",
    right_on="GEOID",
    how="left",
    validate="many_to_one"
)

noaa_spatial = gpd.GeoDataFrame(
    noaa_spatial,
    geometry="geometry",
    crs=counties.crs
)

print("Original county events:", len(county_events))
print("Joined event records:", len(noaa_spatial))
print("Events without geometry:", noaa_spatial.geometry.isna().sum())

Original county events: 49552
Joined event records: 49552
Events without geometry: 0


In [84]:
os.makedirs("data/intermediate", exist_ok=True)

output_file = (
    "data/intermediate/NOAA_Flood_Events_County_Spatial.gpkg"
)

noaa_spatial.to_file(
    output_file,
    layer="flood_events",
    driver="GPKG"
)

print("Saved:", output_file)

Saved: data/intermediate/NOAA_Flood_Events_County_Spatial.gpkg


### ^Exporting as a geodataframe

### now we have a file called data/intermediate/NOAA_Flood_Events_County_Spatial.gpkg With the same as NOAA_Flood_Events_10_States_1996_2026.csv with county geometries for each row

## Now I also want to make a geodataframe that has counties with flood counts per year

In [85]:
county_events = noaa_floods[
    noaa_floods["CZ_TYPE"] == "C"
].copy()

county_events["STATE_FIPS"] = (
    pd.to_numeric(county_events["STATE_FIPS"], errors="coerce")
    .astype("Int64").astype("string").str.zfill(2)
)

county_events["CZ_FIPS"] = (
    pd.to_numeric(county_events["CZ_FIPS"], errors="coerce")
    .astype("Int64").astype("string").str.zfill(3)
)

county_events["GEOID"] = (
    county_events["STATE_FIPS"] + county_events["CZ_FIPS"]
)

county_year_counts = (
    county_events.groupby(["GEOID", "YEAR"])
    .size()
    .reset_index(name="Flood_Count")
)

display(county_year_counts.head())

,GEOID,YEAR,Flood_Count
0,01001,1996,2
1,01001,1998,1
2,01001,1999,2
3,01001,2000,1
4,01001,2001,2


In [86]:
counties["GEOID"] = counties["GEOID"].astype(str)

county_year_gdf = county_year_counts.merge(
    counties,
    on="GEOID",
    how="left",
    validate="many_to_one"
)

county_year_gdf = gpd.GeoDataFrame(
    county_year_gdf,
    geometry="geometry",
    crs=counties.crs
)

print("County-year records:", len(county_year_gdf))
print("Missing geometries:", county_year_gdf.geometry.isna().sum())

display(county_year_gdf.head())

County-year records: 15343
Missing geometries: 0


,GEOID,YEAR,Flood_Count,NAME,STATEFP,COUNTYFP,geometry
0,01001,1996,2,Autauga,01,001,"POLYGON ((-86.58826 32.36775, -86.58834 32.367..."
1,01001,1998,1,Autauga,01,001,"POLYGON ((-86.58826 32.36775, -86.58834 32.367..."
2,01001,1999,2,Autauga,01,001,"POLYGON ((-86.58826 32.36775, -86.58834 32.367..."
3,01001,2000,1,Autauga,01,001,"POLYGON ((-86.58826 32.36775, -86.58834 32.367..."
4,01001,2001,2,Autauga,01,001,"POLYGON ((-86.58826 32.36775, -86.58834 32.367..."


In [87]:
output_file = os.path.join(
    output_folder,
    "NOAA_County_Flood_Counts_By_Year.gpkg"
)

county_year_gdf.to_file(
    output_file,
    layer="county_flood_counts",
    driver="GPKG"
)

print("Saved to:", output_file)

Saved to: data/raw/tiger_counties/NOAA_County_Flood_Counts_By_Year.gpkg


## Now have a countys for my 10 states containing NOAA flood counts per year called Intermediate/NOAA_County_Flood_Counts_By_Year.gpkg

## Now making the "processed" layer for comparison. counts of flood in county for every 10 years within these 10 states (for comparison with gages)

In [88]:
county_events = noaa_floods[
    noaa_floods["CZ_TYPE"].astype(str).str.strip() == "C"
].copy()

county_events["STATE_FIPS"] = (
    pd.to_numeric(county_events["STATE_FIPS"], errors="coerce")
    .astype("Int64").astype("string").str.zfill(2)
)

county_events["CZ_FIPS"] = (
    pd.to_numeric(county_events["CZ_FIPS"], errors="coerce")
    .astype("Int64").astype("string").str.zfill(3)
)

county_events["GEOID"] = (
    county_events["STATE_FIPS"] + county_events["CZ_FIPS"]
)

county_events["YEAR"] = pd.to_numeric(
    county_events["YEAR"], errors="coerce"
)

county_events = county_events.dropna(
    subset=["GEOID", "YEAR"]
)

county_events["YEAR"] = county_events["YEAR"].astype(int)

first_year = county_events["YEAR"].min()
last_year = county_events["YEAR"].max()

print("NOAA data covers:", first_year, "-", last_year)

NOAA data covers: 1996 - 2026


In [89]:
periods = []

start = first_year
end = min(last_year, ((start // 10) + 1) * 10 - 1)

while start <= last_year:
    periods.append({
        "period_start": start,
        "period_end": end,
        "Decade": f"{start}_{end}"
    })

    start = end + 1

    if start <= last_year:
        end = min(last_year, start + 9)

period_df = pd.DataFrame(periods)

display(period_df)

,period_start,period_end,Decade
0,1996,1999,1996_1999
1,2000,2009,2000_2009
2,2010,2019,2010_2019
3,2020,2026,2020_2026


In [90]:
county_events["Decade"] = None

for _, period in period_df.iterrows():
    mask = county_events["YEAR"].between(
        period["period_start"],
        period["period_end"]
    )

    county_events.loc[mask, "Decade"] = period["Decade"]

flood_counts = (
    county_events.groupby(["GEOID", "Decade"])
    .size()
    .reset_index(name="Flood_Count")
)

county_decades = counties[
    ["GEOID", "NAME", "STATEFP", "geometry"]
].merge(period_df, how="cross")

county_decades = county_decades.merge(
    flood_counts,
    on=["GEOID", "Decade"],
    how="left"
)

county_decades["Flood_Count"] = (
    county_decades["Flood_Count"].fillna(0).astype(int)
)

county_decades = gpd.GeoDataFrame(
    county_decades,
    geometry="geometry",
    crs=counties.crs
)

print("Total county-period features:", len(county_decades))
display(county_decades.head())

Total county-period features: 3512


,GEOID,NAME,STATEFP,geometry,period_start,period_end,Decade,Flood_Count
0,40075,Kiowa,40,"POLYGON ((-98.95506 35.11643, -98.94903 35.116...",1996,1999,1996_1999,6
1,40075,Kiowa,40,"POLYGON ((-98.95506 35.11643, -98.94903 35.116...",2000,2009,2000_2009,22
2,40075,Kiowa,40,"POLYGON ((-98.95506 35.11643, -98.94903 35.116...",2010,2019,2010_2019,14
3,40075,Kiowa,40,"POLYGON ((-98.95506 35.11643, -98.94903 35.116...",2020,2026,2020_2026,0
4,40023,Choctaw,40,"POLYGON ((-95.99158 34.1568, -95.99154 34.1568...",1996,1999,1996_1999,7


In [91]:

county_decades = county_decades.copy()

# Remove missing or empty geometries
county_decades = county_decades[
    county_decades.geometry.notna() &
    ~county_decades.geometry.is_empty
].copy()

# Convert all polygons to MultiPolygon
def to_multipolygon(geom):
    if geom.geom_type == "Polygon":
        return MultiPolygon([geom])
    elif geom.geom_type == "MultiPolygon":
        return geom
    else:
        return None

county_decades["geometry"] = (
    county_decades.geometry.apply(to_multipolygon)
)

county_decades = county_decades[
    county_decades.geometry.notna()
].copy()

print(county_decades.geom_type.value_counts())

MultiPolygon    3512
Name: count, dtype: int64


In [92]:
output_folder = os.path.join(
    project_dir,
    "data",
    "processed",
    "NOAA_Floods_By_State"
)

os.makedirs(output_folder, exist_ok=True)

In [93]:
state_names = {
    "01": "Alabama",
    "05": "Arkansas",
    "19": "Iowa",
    "21": "Kentucky",
    "28": "Mississippi",
    "29": "Missouri",
    "31": "Nebraska",
    "40": "Oklahoma",
    "47": "Tennessee",
    "54": "West_Virginia"
}

for state_fips, state_name in state_names.items():

    print(f"\nProcessing {state_name}...")

    state_data = county_decades[
        county_decades["STATEFP"].astype(str).str.zfill(2)
        == state_fips
    ].copy()

    gdb_name = f"{state_name}_Floods.gdb"
    gdb_path = os.path.join(output_folder, gdb_name)

    if os.path.exists(gdb_path):
        shutil.rmtree(gdb_path)

    for i, period in period_df.iterrows():

        decade = period["Decade"]

        layer = state_data[
            state_data["Decade"] == decade
        ].copy()

        layer_name = f"Floods_{decade}"

        if layer.empty:
            print(f"  Skipping {layer_name}: no county features")
            continue

        pyogrio.write_dataframe(
            layer.to_crs(epsg=state_epsg[state_fips]),
            gdb_path,
            layer=layer_name,
            driver="OpenFileGDB",
            geometry_type="MultiPolygon",
            promote_to_multi=True
        )

        print(f"  Saved {layer_name}: {len(layer)} counties")

print("\nAll state geodatabases created!")
print("Output folder:", output_folder)


Processing Alabama...
  Saved Floods_1996_1999: 67 counties
  Saved Floods_2000_2009: 67 counties


/opt/anaconda3/envs/geospatial/lib/python3.14/site-packages/pyogrio/raw.py:723: RuntimeWarning: Field period_start of type Integer64 will be written as a Float64. To get Integer64, use layer creation option TARGET_ARCGIS_VERSION=ARCGIS_PRO_3_2_OR_LATER
  ogr_write(
/opt/anaconda3/envs/geospatial/lib/python3.14/site-packages/pyogrio/raw.py:723: RuntimeWarning: Field period_end of type Integer64 will be written as a Float64. To get Integer64, use layer creation option TARGET_ARCGIS_VERSION=ARCGIS_PRO_3_2_OR_LATER
  ogr_write(
/opt/anaconda3/envs/geospatial/lib/python3.14/site-packages/pyogrio/raw.py:723: RuntimeWarning: Field Flood_Count of type Integer64 will be written as a Float64. To get Integer64, use layer creation option TARGET_ARCGIS_VERSION=ARCGIS_PRO_3_2_OR_LATER
  ogr_write(
/opt/anaconda3/envs/geospatial/lib/python3.14/site-packages/pyogrio/raw.py:723: RuntimeWarning: Field period_start of type Integer64 will be written as a Float64. To get Integer64, use layer creation optio

  Saved Floods_2010_2019: 67 counties
  Saved Floods_2020_2026: 67 counties

Processing Arkansas...
  Saved Floods_1996_1999: 75 counties


/opt/anaconda3/envs/geospatial/lib/python3.14/site-packages/pyogrio/raw.py:723: RuntimeWarning: Field period_start of type Integer64 will be written as a Float64. To get Integer64, use layer creation option TARGET_ARCGIS_VERSION=ARCGIS_PRO_3_2_OR_LATER
  ogr_write(
/opt/anaconda3/envs/geospatial/lib/python3.14/site-packages/pyogrio/raw.py:723: RuntimeWarning: Field period_end of type Integer64 will be written as a Float64. To get Integer64, use layer creation option TARGET_ARCGIS_VERSION=ARCGIS_PRO_3_2_OR_LATER
  ogr_write(
/opt/anaconda3/envs/geospatial/lib/python3.14/site-packages/pyogrio/raw.py:723: RuntimeWarning: Field Flood_Count of type Integer64 will be written as a Float64. To get Integer64, use layer creation option TARGET_ARCGIS_VERSION=ARCGIS_PRO_3_2_OR_LATER
  ogr_write(
/opt/anaconda3/envs/geospatial/lib/python3.14/site-packages/pyogrio/raw.py:723: RuntimeWarning: Field period_start of type Integer64 will be written as a Float64. To get Integer64, use layer creation optio

  Saved Floods_2000_2009: 75 counties
  Saved Floods_2010_2019: 75 counties
  Saved Floods_2020_2026: 75 counties

Processing Iowa...
  Saved Floods_1996_1999: 99 counties


/opt/anaconda3/envs/geospatial/lib/python3.14/site-packages/pyogrio/raw.py:723: RuntimeWarning: Field period_start of type Integer64 will be written as a Float64. To get Integer64, use layer creation option TARGET_ARCGIS_VERSION=ARCGIS_PRO_3_2_OR_LATER
  ogr_write(
/opt/anaconda3/envs/geospatial/lib/python3.14/site-packages/pyogrio/raw.py:723: RuntimeWarning: Field period_end of type Integer64 will be written as a Float64. To get Integer64, use layer creation option TARGET_ARCGIS_VERSION=ARCGIS_PRO_3_2_OR_LATER
  ogr_write(
/opt/anaconda3/envs/geospatial/lib/python3.14/site-packages/pyogrio/raw.py:723: RuntimeWarning: Field Flood_Count of type Integer64 will be written as a Float64. To get Integer64, use layer creation option TARGET_ARCGIS_VERSION=ARCGIS_PRO_3_2_OR_LATER
  ogr_write(
/opt/anaconda3/envs/geospatial/lib/python3.14/site-packages/pyogrio/raw.py:723: RuntimeWarning: Field period_start of type Integer64 will be written as a Float64. To get Integer64, use layer creation optio

... (8 outputs left out)

  Saved Floods_2000_2009: 93 counties
  Saved Floods_2010_2019: 93 counties
  Saved Floods_2020_2026: 93 counties

Processing Oklahoma...
  Saved Floods_1996_1999: 77 counties
  Saved Floods_2000_2009: 77 counties
  Saved Floods_2010_2019: 77 counties


/opt/anaconda3/envs/geospatial/lib/python3.14/site-packages/pyogrio/raw.py:723: RuntimeWarning: Field period_start of type Integer64 will be written as a Float64. To get Integer64, use layer creation option TARGET_ARCGIS_VERSION=ARCGIS_PRO_3_2_OR_LATER
  ogr_write(
/opt/anaconda3/envs/geospatial/lib/python3.14/site-packages/pyogrio/raw.py:723: RuntimeWarning: Field period_end of type Integer64 will be written as a Float64. To get Integer64, use layer creation option TARGET_ARCGIS_VERSION=ARCGIS_PRO_3_2_OR_LATER
  ogr_write(
/opt/anaconda3/envs/geospatial/lib/python3.14/site-packages/pyogrio/raw.py:723: RuntimeWarning: Field Flood_Count of type Integer64 will be written as a Float64. To get Integer64, use layer creation option TARGET_ARCGIS_VERSION=ARCGIS_PRO_3_2_OR_LATER
  ogr_write(
/opt/anaconda3/envs/geospatial/lib/python3.14/site-packages/pyogrio/raw.py:723: RuntimeWarning: Field period_start of type Integer64 will be written as a Float64. To get Integer64, use layer creation optio

  Saved Floods_2020_2026: 77 counties

Processing Tennessee...
  Saved Floods_1996_1999: 95 counties
  Saved Floods_2000_2009: 95 counties
  Saved Floods_2010_2019: 95 counties
  Saved Floods_2020_2026: 95 counties

Processing West_Virginia...
  Saved Floods_1996_1999: 55 counties
  Saved Floods_2000_2009: 55 counties
  Saved Floods_2010_2019: 55 counties
  Saved Floods_2020_2026: 55 counties

All state geodatabases created!
Output folder: /Users/sophiameyers/Documents/GitHub/USGS-Gages-Research/data/processed/NOAA_Floods_By_State


/opt/anaconda3/envs/geospatial/lib/python3.14/site-packages/pyogrio/raw.py:723: RuntimeWarning: Field period_start of type Integer64 will be written as a Float64. To get Integer64, use layer creation option TARGET_ARCGIS_VERSION=ARCGIS_PRO_3_2_OR_LATER
  ogr_write(
/opt/anaconda3/envs/geospatial/lib/python3.14/site-packages/pyogrio/raw.py:723: RuntimeWarning: Field period_end of type Integer64 will be written as a Float64. To get Integer64, use layer creation option TARGET_ARCGIS_VERSION=ARCGIS_PRO_3_2_OR_LATER
  ogr_write(
/opt/anaconda3/envs/geospatial/lib/python3.14/site-packages/pyogrio/raw.py:723: RuntimeWarning: Field Flood_Count of type Integer64 will be written as a Float64. To get Integer64, use layer creation option TARGET_ARCGIS_VERSION=ARCGIS_PRO_3_2_OR_LATER
  ogr_write(
/opt/anaconda3/envs/geospatial/lib/python3.14/site-packages/pyogrio/raw.py:723: RuntimeWarning: Field period_start of type Integer64 will be written as a Float64. To get Integer64, use layer creation optio

In [94]:
for state_name in state_names.values():

    gdb_path = os.path.join(
        output_folder,
        f"{state_name}_Floods.gdb"
    )

    print(f"\n{state_name}:")
    print(pyogrio.list_layers(gdb_path))


Alabama:
[['Floods_1996_1999' 'MultiPolygon']
 ['Floods_2000_2009' 'MultiPolygon']
 ['Floods_2010_2019' 'MultiPolygon']
 ['Floods_2020_2026' 'MultiPolygon']]

Arkansas:
[['Floods_1996_1999' 'MultiPolygon']
 ['Floods_2000_2009' 'MultiPolygon']
 ['Floods_2010_2019' 'MultiPolygon']
 ['Floods_2020_2026' 'MultiPolygon']]

Iowa:
[['Floods_1996_1999' 'MultiPolygon']
 ['Floods_2000_2009' 'MultiPolygon']
 ['Floods_2010_2019' 'MultiPolygon']
 ['Floods_2020_2026' 'MultiPolygon']]

Kentucky:
[['Floods_1996_1999' 'MultiPolygon']
 ['Floods_2000_2009' 'MultiPolygon']
 ['Floods_2010_2019' 'MultiPolygon']
 ['Floods_2020_2026' 'MultiPolygon']]

Mississippi:
[['Floods_1996_1999' 'MultiPolygon']
 ['Floods_2000_2009' 'MultiPolygon']
 ['Floods_2010_2019' 'MultiPolygon']
 ['Floods_2020_2026' 'MultiPolygon']]

Missouri:
[['Floods_1996_1999' 'MultiPolygon']
 ['Floods_2000_2009' 'MultiPolygon']
 ['Floods_2010_2019' 'MultiPolygon']
 ['Floods_2020_2026' 'MultiPolygon']]

Nebraska:
[['Floods_1996_1999' 'MultiPoly

### ^Now have a geodatabase with flood agregated by county that folows the time frame of the gages gdb called /processed/NOAA_Flood_By_State.gdb 

# FEMA Presidential flood mapping
### Copying the dame workflow to get the same outputs with the files I already have /intermediate/fema_disaster_declarations_floods_v2.csv and /intermediate/fema_flood_declaration_counts_by_county_year.csv

In [95]:
counts_file = os.path.join(
    project_dir,
    "data",
    "intermediate",
    "fema_flood_declaration_counts_by_county_year.csv"
)

events_file = os.path.join(
    project_dir,
    "data",
    "raw",
    "fema_disaster_declarations_floods_v2.csv"
)

county_base = counties.copy()

county_base["GEOID"] = county_base["GEOID"].astype(str).str.zfill(5)
county_base["STATEFP"] = county_base["STATEFP"].astype(str).str.zfill(2)

state_fips = [
    "01", "05", "19", "21", "28",
    "29", "31", "40", "47", "54"
]

county_base = county_base[
    county_base["STATEFP"].isin(state_fips)
].copy()

print("County polygons:", len(county_base))

County polygons: 878


In [96]:
fema_counts = pd.read_csv(counts_file)

fema_counts["STATEFP"] = (
    pd.to_numeric(fema_counts["fipsStateCode"], errors="coerce")
    .astype("Int64").astype("string").str.zfill(2)
)

fema_counts["COUNTYFP"] = (
    pd.to_numeric(fema_counts["fipsCountyCode"], errors="coerce")
    .astype("Int64").astype("string").str.zfill(3)
)

fema_counts["GEOID"] = (
    fema_counts["STATEFP"] + fema_counts["COUNTYFP"]
)

fema_counts["year"] = pd.to_numeric(
    fema_counts["year"], errors="coerce"
)

fema_counts["flood_declaration_count"] = pd.to_numeric(
    fema_counts["flood_declaration_count"], errors="coerce"
)

fema_counts = fema_counts[
    fema_counts["STATEFP"].isin(state_fips)
].dropna(
    subset=["GEOID", "year", "flood_declaration_count"]
).copy()

fema_counts["year"] = fema_counts["year"].astype(int)

annual_counts = (
    fema_counts.groupby(["GEOID", "year"])
    ["flood_declaration_count"]
    .sum()
    .reset_index(name="Flood_Count")
)

print("County-year records:", len(annual_counts))

County-year records: 3144


In [97]:
years = range(
    annual_counts["year"].min(),
    annual_counts["year"].max() + 1
)

county_years = county_base[
    ["GEOID", "NAME", "STATEFP", "geometry"]
].merge(
    pd.DataFrame({"year": list(years)}),
    how="cross"
)

fema_county_year = county_years.merge(
    annual_counts,
    on=["GEOID", "year"],
    how="left",
    validate="one_to_one"
)

fema_county_year["Flood_Count"] = (
    fema_county_year["Flood_Count"].fillna(0).astype(int)
)

fema_county_year = gpd.GeoDataFrame(
    fema_county_year,
    geometry="geometry",
    crs=county_base.crs
)

print("Total county-year features:", len(fema_county_year))
display(fema_county_year.head())

Total county-year features: 54436


,GEOID,NAME,STATEFP,geometry,year,Flood_Count
0,40075,Kiowa,40,"POLYGON ((-98.95506 35.11643, -98.94903 35.116...",1965,0
1,40075,Kiowa,40,"POLYGON ((-98.95506 35.11643, -98.94903 35.116...",1966,0
2,40075,Kiowa,40,"POLYGON ((-98.95506 35.11643, -98.94903 35.116...",1967,0
3,40075,Kiowa,40,"POLYGON ((-98.95506 35.11643, -98.94903 35.116...",1968,0
4,40075,Kiowa,40,"POLYGON ((-98.95506 35.11643, -98.94903 35.116...",1969,0


In [98]:
intermediate_folder = os.path.join(
    project_dir, "data", "intermediate"
)

os.makedirs(intermediate_folder, exist_ok=True)

annual_output = os.path.join(
    intermediate_folder,
    "FEMA_County_Flood_Counts_By_Year.gpkg"
)

if os.path.exists(annual_output):
    os.remove(annual_output)

pyogrio.write_dataframe(
    fema_county_year,
    annual_output,
    layer="county_flood_counts",
    driver="GPKG"
)

print("Saved:", annual_output)

Saved: /Users/sophiameyers/Documents/GitHub/USGS-Gages-Research/data/intermediate/FEMA_County_Flood_Counts_By_Year.gpkg


### ^^ Now have a counts per county per year geodataframe layer for fema called intermediate/FEMA_County_Flood_Counts_By_Year.gpkg

### creating a file amending county spatial data to the original csv (county shape per row/flood event)

In [99]:
fema_events = pd.read_csv(events_file, low_memory=False)

fema_events["STATEFP"] = (
    pd.to_numeric(fema_events["fipsStateCode"], errors="coerce")
    .astype("Int64").astype("string").str.zfill(2)
)

fema_events["COUNTYFP"] = (
    pd.to_numeric(fema_events["fipsCountyCode"], errors="coerce")
    .astype("Int64").astype("string").str.zfill(3)
)

fema_events["GEOID"] = (
    fema_events["STATEFP"] + fema_events["COUNTYFP"]
)

fema_events = fema_events[
    fema_events["STATEFP"].isin(state_fips) &
    fema_events["COUNTYFP"].notna() &
    (fema_events["COUNTYFP"] != "000")
].copy()

fema_events["year"] = (
    pd.to_datetime(
        fema_events["declarationDate"],
        errors="coerce",
        utc=True
    ).dt.year
)

fema_events = fema_events.dropna(subset=["year"]).copy()
fema_events["year"] = fema_events["year"].astype(int)

print("County declaration records:", len(fema_events))

County declaration records: 3435


In [100]:
fema_spatial = fema_events.merge(
    county_base[["GEOID", "NAME", "geometry"]],
    on="GEOID",
    how="left",
    validate="many_to_one",
    indicator=True
)

print("Matched records:", (fema_spatial["_merge"] == "both").sum())
print("Unmatched records:", (fema_spatial["_merge"] == "left_only").sum())

fema_spatial = fema_spatial[
    fema_spatial["_merge"] == "both"
].drop(columns="_merge")

fema_spatial = gpd.GeoDataFrame(
    fema_spatial,
    geometry="geometry",
    crs=county_base.crs
)

display(fema_spatial.head())

Matched records: 3435
Unmatched records: 0


,femaDeclarationString,disasterNumber,state,declarationType,declarationDate,fyDeclared,incidentType,declarationTitle,ihProgramDeclared,iaProgramDeclared,...,designatedIncidentTypes,lastRefresh,hash,id,STATEFP,COUNTYFP,GEOID,year,NAME,geometry
0,DR-4884-WV,4884,WV,DR,2025-07-22T00:00:00.000Z,2025,Flood,"SEVERE STORMS, STRAIGHT-LINE WINDS, FLOODING, ...",True,False,...,"2,F,M,W",2026-10-05T11:26:24.661Z,a2879fa82f857b1be298286dae78ba8bb8c77ea2,dd4ff5a5-1f32-4f5a-b599-aeaf655eb550,54,049,54049,2025,Marion,"POLYGON ((-80.08094 39.55094, -80.08083 39.550..."
1,DR-4884-WV,4884,WV,DR,2025-07-22T00:00:00.000Z,2025,Flood,"SEVERE STORMS, STRAIGHT-LINE WINDS, FLOODING, ...",True,False,...,"2,F,M,W",2026-10-05T11:26:24.661Z,dc94a198fd4a1b1e09dbb81a1200d755020ac4a4,b7661e14-20a6-4a0b-9be3-7e2d84da7f3d,54,069,54069,2025,Ohio,"POLYGON ((-80.7338 40.03341, -80.7337 40.03359..."
2,DR-4873-AR,4873,AR,DR,2025-05-21T00:00:00.000Z,2025,Flood,"SEVERE STORMS, TORNADOES, AND FLOODING",False,False,...,"F,T,W",2026-10-05T11:26:24.661Z,19d63049b78d392db0933ccc8829303c8f1bc81c,6ab60c41-f2c8-4039-b580-e313c98703a5,05,085,05085,2025,Lonoke,"POLYGON ((-91.8908 35.03086, -91.89068 35.0307..."
3,DR-4873-AR,4873,AR,DR,2025-05-21T00:00:00.000Z,2025,Flood,"SEVERE STORMS, TORNADOES, AND FLOODING",True,False,...,"F,T,W",2026-10-05T11:26:24.661Z,66d4c6a52e0fdf7a69f8c39db10317d43bc70494,3b232232-a077-4e90-862d-69befeeeb278,05,091,05091,2025,Miller,"POLYGON ((-93.83833 33.14852, -93.83852 33.148..."
4,DR-4873-AR,4873,AR,DR,2025-05-21T00:00:00.000Z,2025,Flood,"SEVERE STORMS, TORNADOES, AND FLOODING",True,False,...,"F,T,W",2026-10-05T11:26:24.661Z,05fc30c67af1465276755bbe0197c059958fbafe,9f4ac62e-c462-4f61-8fd0-61df73bc1d82,05,055,05055,2025,Greene,"POLYGON ((-90.31975 36.239, -90.31975 36.23015..."


In [101]:
events_output = os.path.join(
    intermediate_folder,
    "FEMA_Flood_Events_County_Spatial.gpkg"
)

if os.path.exists(events_output):
    os.remove(events_output)

pyogrio.write_dataframe(
    fema_spatial,
    events_output,
    layer="fema_flood_events",
    driver="GPKG"
)

print("Saved:", events_output)

Saved: /Users/sophiameyers/Documents/GitHub/USGS-Gages-Research/data/intermediate/FEMA_Flood_Events_County_Spatial.gpkg


## ^^Now have original fema flood event rows with county geometry amended per row called intermediate/FEMA_Flood_Events_County_Spatial.gpkg

## Now creating the gdb for the devided decades in our 10 study states with FEMA

In [102]:
def make_periods(first_year, last_year):

    periods = []

    start = first_year
    end = min(last_year, ((start // 10) + 1) * 10 - 1)

    while start <= last_year:

        periods.append({
            "period_start": start,
            "period_end": end,
            "Decade": f"{start}_{end}"
        })

        start = end + 1

        if start <= last_year:
            end = min(last_year, start + 9)

    return pd.DataFrame(periods)


count_periods = make_periods(
    fema_counts["year"].min(),
    fema_counts["year"].max()
)

event_periods = make_periods(
    fema_events["year"].min(),
    fema_events["year"].max()
)

display(count_periods)
display(event_periods)

,period_start,period_end,Decade
0,1965,1969,1965_1969
1,1970,1979,1970_1979
2,1980,1989,1980_1989
3,1990,1999,1990_1999
4,2000,2009,2000_2009
5,2010,2019,2010_2019
6,2020,2026,2020_2026


,period_start,period_end,Decade
0,1965,1969,1965_1969
1,1970,1979,1970_1979
2,1980,1989,1980_1989
3,1990,1999,1990_1999
4,2000,2009,2000_2009
5,2010,2019,2010_2019
6,2020,2026,2020_2026


In [103]:
fema_counts["Decade"] = None

for _, period in count_periods.iterrows():

    mask = fema_counts["year"].between(
        period["period_start"],
        period["period_end"]
    )

    fema_counts.loc[mask, "Decade"] = period["Decade"]


fema_decade_counts = (
    fema_counts.groupby(["GEOID", "Decade"])
    ["flood_declaration_count"]
    .sum()
    .reset_index(name="Flood_Count")
)

# Create every county for every period, including zero counts
county_decade_frames = []

for _, period in count_periods.iterrows():

    decade = period["Decade"]

    layer = county_base.copy()
    layer["Decade"] = decade

    layer = layer.merge(
        fema_decade_counts[
            fema_decade_counts["Decade"] == decade
        ][["GEOID", "Flood_Count"]],
        on="GEOID",
        how="left"
    )

    layer["Flood_Count"] = (
        layer["Flood_Count"].fillna(0).astype(int)
    )

    county_decade_frames.append(layer)

fema_county_decades = gpd.GeoDataFrame(
    pd.concat(county_decade_frames, ignore_index=True),
    geometry="geometry",
    crs=county_base.crs
)

print("County-decade features:", len(fema_county_decades))

County-decade features: 6146


In [104]:
state_names = {
    "01": "Alabama",
    "05": "Arkansas",
    "19": "Iowa",
    "21": "Kentucky",
    "28": "Mississippi",
    "29": "Missouri",
    "31": "Nebraska",
    "40": "Oklahoma",
    "47": "Tennessee",
    "54": "West_Virginia"
}

output_folder = os.path.join(
    project_dir,
    "data",
    "processed",
    "FEMA_Floods_By_State"
)

os.makedirs(output_folder, exist_ok=True)

In [105]:
for state_fips, state_name in state_names.items():

    print(f"\nProcessing {state_name}...")

    gdb_path = os.path.join(
        output_folder,
        f"{state_name}_Floods.gdb"
    )

    if os.path.exists(gdb_path):
        shutil.rmtree(gdb_path)

    # -------------------------------
    # COUNTY-DECADE FLOOD COUNT LAYERS
    # -------------------------------

    state_counts = fema_county_decades[
        fema_county_decades["STATEFP"].astype(str).str.zfill(2)
        == state_fips
    ].copy()

    for _, period in count_periods.iterrows():

        decade = period["Decade"]

        layer = state_counts[
            state_counts["Decade"] == decade
        ].copy()

        layer = layer[
            layer.geometry.notna() &
            ~layer.geometry.is_empty
        ].copy()

        pyogrio.write_dataframe(
            layer.to_crs(epsg=state_epsg[state_fips]),
            gdb_path,
            layer=f"FEMA_Counts_{decade}",
            driver="OpenFileGDB",
            geometry_type="MultiPolygon",
            promote_to_multi=True
        )

        print(f"  Saved FEMA_Counts_{decade}: {len(layer)} counties")

    # -------------------------------
    # INDIVIDUAL DECLARATION LAYERS
    # -------------------------------

    state_events = fema_spatial[
        fema_spatial["STATEFP"].astype(str).str.zfill(2)
        == state_fips
    ].copy()

    state_events["Decade"] = None

    for _, period in event_periods.iterrows():

        mask = state_events["year"].between(
            period["period_start"],
            period["period_end"]
        )

        state_events.loc[mask, "Decade"] = period["Decade"]

    for _, period in event_periods.iterrows():

        decade = period["Decade"]

        layer = state_events[
            state_events["Decade"] == decade
        ].copy()

        layer = layer[
            layer.geometry.notna() &
            ~layer.geometry.is_empty
        ].copy()

        if layer.empty:
            continue

        pyogrio.write_dataframe(
            layer.to_crs(epsg=state_epsg[state_fips]),
            gdb_path,
            layer=f"FEMA_Decl_{decade}",
            driver="OpenFileGDB",
            geometry_type="MultiPolygon",
            promote_to_multi=True
        )

        print(
            f"  Saved FEMA_Decl_{decade}: "
            f"{len(layer)} declaration records"
        )

print("\nAll FEMA geodatabases created!")
print("Output folder:", output_folder)


Processing Alabama...


/opt/anaconda3/envs/geospatial/lib/python3.14/site-packages/pyogrio/raw.py:723: RuntimeWarning: Field Flood_Count of type Integer64 will be written as a Float64. To get Integer64, use layer creation option TARGET_ARCGIS_VERSION=ARCGIS_PRO_3_2_OR_LATER
  ogr_write(
/opt/anaconda3/envs/geospatial/lib/python3.14/site-packages/pyogrio/raw.py:723: RuntimeWarning: Field Flood_Count of type Integer64 will be written as a Float64. To get Integer64, use layer creation option TARGET_ARCGIS_VERSION=ARCGIS_PRO_3_2_OR_LATER
  ogr_write(
/opt/anaconda3/envs/geospatial/lib/python3.14/site-packages/pyogrio/raw.py:723: RuntimeWarning: Field Flood_Count of type Integer64 will be written as a Float64. To get Integer64, use layer creation option TARGET_ARCGIS_VERSION=ARCGIS_PRO_3_2_OR_LATER
  ogr_write(
/opt/anaconda3/envs/geospatial/lib/python3.14/site-packages/pyogrio/raw.py:723: RuntimeWarning: Field Flood_Count of type Integer64 will be written as a Float64. To get Integer64, use layer creation option

  Saved FEMA_Counts_1965_1969: 67 counties
  Saved FEMA_Counts_1970_1979: 67 counties
  Saved FEMA_Counts_1980_1989: 67 counties
  Saved FEMA_Counts_1990_1999: 67 counties
  Saved FEMA_Counts_2000_2009: 67 counties


/opt/anaconda3/envs/geospatial/lib/python3.14/site-packages/pyogrio/raw.py:723: RuntimeWarning: Field Flood_Count of type Integer64 will be written as a Float64. To get Integer64, use layer creation option TARGET_ARCGIS_VERSION=ARCGIS_PRO_3_2_OR_LATER
  ogr_write(
/opt/anaconda3/envs/geospatial/lib/python3.14/site-packages/pyogrio/raw.py:723: RuntimeWarning: Field Flood_Count of type Integer64 will be written as a Float64. To get Integer64, use layer creation option TARGET_ARCGIS_VERSION=ARCGIS_PRO_3_2_OR_LATER
  ogr_write(
/opt/anaconda3/envs/geospatial/lib/python3.14/site-packages/pyogrio/raw.py:723: RuntimeWarning: Field disasterNumber of type Integer64 will be written as a Float64. To get Integer64, use layer creation option TARGET_ARCGIS_VERSION=ARCGIS_PRO_3_2_OR_LATER
  ogr_write(
/opt/anaconda3/envs/geospatial/lib/python3.14/site-packages/pyogrio/raw.py:723: RuntimeWarning: Field fyDeclared of type Integer64 will be written as a Float64. To get Integer64, use layer creation opti

  Saved FEMA_Counts_2010_2019: 67 counties
  Saved FEMA_Counts_2020_2026: 67 counties
  Saved FEMA_Decl_1970_1979: 90 declaration records
  Saved FEMA_Decl_1980_1989: 1 declaration records
  Saved FEMA_Decl_1990_1999: 12 declaration records

Processing Arkansas...


/opt/anaconda3/envs/geospatial/lib/python3.14/site-packages/pyogrio/raw.py:723: RuntimeWarning: Field Flood_Count of type Integer64 will be written as a Float64. To get Integer64, use layer creation option TARGET_ARCGIS_VERSION=ARCGIS_PRO_3_2_OR_LATER
  ogr_write(
/opt/anaconda3/envs/geospatial/lib/python3.14/site-packages/pyogrio/raw.py:723: RuntimeWarning: Field Flood_Count of type Integer64 will be written as a Float64. To get Integer64, use layer creation option TARGET_ARCGIS_VERSION=ARCGIS_PRO_3_2_OR_LATER
  ogr_write(
/opt/anaconda3/envs/geospatial/lib/python3.14/site-packages/pyogrio/raw.py:723: RuntimeWarning: Field Flood_Count of type Integer64 will be written as a Float64. To get Integer64, use layer creation option TARGET_ARCGIS_VERSION=ARCGIS_PRO_3_2_OR_LATER
  ogr_write(


... (44 outputs left out)

  Saved FEMA_Counts_1990_1999: 55 counties
  Saved FEMA_Counts_2000_2009: 55 counties
  Saved FEMA_Counts_2010_2019: 55 counties
  Saved FEMA_Counts_2020_2026: 55 counties
  Saved FEMA_Decl_1965_1969: 37 declaration records


/opt/anaconda3/envs/geospatial/lib/python3.14/site-packages/pyogrio/raw.py:723: RuntimeWarning: Field disasterNumber of type Integer64 will be written as a Float64. To get Integer64, use layer creation option TARGET_ARCGIS_VERSION=ARCGIS_PRO_3_2_OR_LATER
  ogr_write(
/opt/anaconda3/envs/geospatial/lib/python3.14/site-packages/pyogrio/raw.py:723: RuntimeWarning: Field fyDeclared of type Integer64 will be written as a Float64. To get Integer64, use layer creation option TARGET_ARCGIS_VERSION=ARCGIS_PRO_3_2_OR_LATER
  ogr_write(
/opt/anaconda3/envs/geospatial/lib/python3.14/site-packages/pyogrio/raw.py:723: RuntimeWarning: Field fipsStateCode of type Integer64 will be written as a Float64. To get Integer64, use layer creation option TARGET_ARCGIS_VERSION=ARCGIS_PRO_3_2_OR_LATER
  ogr_write(
/opt/anaconda3/envs/geospatial/lib/python3.14/site-packages/pyogrio/raw.py:723: RuntimeWarning: Field fipsCountyCode of type Integer64 will be written as a Float64. To get Integer64, use layer creation

  Saved FEMA_Decl_1970_1979: 56 declaration records
  Saved FEMA_Decl_1980_1989: 48 declaration records
  Saved FEMA_Decl_1990_1999: 61 declaration records
  Saved FEMA_Decl_2000_2009: 26 declaration records
  Saved FEMA_Decl_2010_2019: 50 declaration records
  Saved FEMA_Decl_2020_2026: 40 declaration records

All FEMA geodatabases created!
Output folder: /Users/sophiameyers/Documents/GitHub/USGS-Gages-Research/data/processed/FEMA_Floods_By_State


/opt/anaconda3/envs/geospatial/lib/python3.14/site-packages/pyogrio/raw.py:723: RuntimeWarning: Field disasterNumber of type Integer64 will be written as a Float64. To get Integer64, use layer creation option TARGET_ARCGIS_VERSION=ARCGIS_PRO_3_2_OR_LATER
  ogr_write(
/opt/anaconda3/envs/geospatial/lib/python3.14/site-packages/pyogrio/raw.py:723: RuntimeWarning: Field fyDeclared of type Integer64 will be written as a Float64. To get Integer64, use layer creation option TARGET_ARCGIS_VERSION=ARCGIS_PRO_3_2_OR_LATER
  ogr_write(
/opt/anaconda3/envs/geospatial/lib/python3.14/site-packages/pyogrio/raw.py:723: RuntimeWarning: Field fipsStateCode of type Integer64 will be written as a Float64. To get Integer64, use layer creation option TARGET_ARCGIS_VERSION=ARCGIS_PRO_3_2_OR_LATER
  ogr_write(
/opt/anaconda3/envs/geospatial/lib/python3.14/site-packages/pyogrio/raw.py:723: RuntimeWarning: Field fipsCountyCode of type Integer64 will be written as a Float64. To get Integer64, use layer creation

## ^^Now have a gdb structured like our gage activity gdb per county per called decade processed/FEMA_Floods_By_State


### Fixing projection to NAD 83 in meters, no verticle crs, and proper UTM per state

In [106]:
for state_fips, state_name in state_names.items():

    epsg = state_epsg[state_fips]

    print(f"\n{state_name} | Expected EPSG:{epsg}")

    # Example: FEMA
    gdb_path = os.path.join(
        project_dir,
        "data",
        "processed",
        "NOAA_Floods_By_State",
        f"{state_name}_Floods.gdb"
    )

    for layer_name, geometry_type in pyogrio.list_layers(gdb_path):

        info = pyogrio.read_info(
            gdb_path,
            layer=layer_name
        )

        print(
            layer_name,
            "| CRS:", info["crs"],
            "| Geometry:", geometry_type
        )


Alabama | Expected EPSG:26916
Floods_1996_1999 | CRS: EPSG:26916 | Geometry: MultiPolygon
Floods_2000_2009 | CRS: EPSG:26916 | Geometry: MultiPolygon
Floods_2010_2019 | CRS: EPSG:26916 | Geometry: MultiPolygon
Floods_2020_2026 | CRS: EPSG:26916 | Geometry: MultiPolygon

Arkansas | Expected EPSG:26915
Floods_1996_1999 | CRS: EPSG:26915 | Geometry: MultiPolygon
Floods_2000_2009 | CRS: EPSG:26915 | Geometry: MultiPolygon
Floods_2010_2019 | CRS: EPSG:26915 | Geometry: MultiPolygon
Floods_2020_2026 | CRS: EPSG:26915 | Geometry: MultiPolygon

Iowa | Expected EPSG:26915
Floods_1996_1999 | CRS: EPSG:26915 | Geometry: MultiPolygon
Floods_2000_2009 | CRS: EPSG:26915 | Geometry: MultiPolygon
Floods_2010_2019 | CRS: EPSG:26915 | Geometry: MultiPolygon
Floods_2020_2026 | CRS: EPSG:26915 | Geometry: MultiPolygon

Kentucky | Expected EPSG:26916
Floods_1996_1999 | CRS: EPSG:26916 | Geometry: MultiPolygon
Floods_2000_2009 | CRS: EPSG:26916 | Geometry: MultiPolygon
Floods_2010_2019 | CRS: EPSG:26916 | 